# About Dataset
Business Intelligence Program Strategy — Student Success Optimization
Background

Walsoft Computer Institute runs a Business Intelligence (BI) training program for students from diverse educational, geographical, and demographic backgrounds. The institute has collected detailed data on student attributes, entry exams, study effort, and final performance in two technical subjects: Python Programming and Database Systems.

As part of an internal review, the leadership team has hired you — a Data Science Consultant — to analyze this dataset and provide clear, evidence-based recommendations on how to improve:

    Admissions decision-making
    Academic support strategies
    Overall program imnd ROI
e

## Your Mission

Answer this central question:

    “Using the BI program dataset, how can Walsoft strategically improve student success, optimize resources, and increase the effectiveness of its training program?”


***

In [18]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from scipy.stats import f_oneway
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

In [19]:
dataset = pd.read_csv(
    "https://raw.githubusercontent.com/BrianKvin/sample_data/main/bi.csv",
    encoding="latin1",
    on_bad_lines="skip"
)
dataset.head()

,fNAME,lNAME,Age,gender,country,residence,entryEXAM,prevEducation,studyHOURS,Python,DB
0,Christina,Binger,44,Female,Norway,Private,72,Masters,158,59.0,55
1,Alex,Walekhwa,60,M,Kenya,Private,79,Diploma,150,60.0,75
2,Philip,Leo,25,Male,Uganda,Sognsvann,55,HighSchool,130,74.0,50
3,Shoni,Hlongwane,22,F,Rsa,Sognsvann,40,High School,120,NaN,44
4,Maria,Kedibone,23,Female,South Africa,Sognsvann,65,High School,122,91.0,80


***


# Data Cleaning imputation strategy


## 1. Correcting Country Name Inconconsistencies
Country name inconsistencies
e.g. Norge → Norway, RSA → South Africa, UK → United Kingdom

In [23]:
dataset['country'] = dataset['country'].str.strip().str.title()
country_map = {
    "Norge": "Norway",
    "Rsa": "South Africa",
    "Uk": "United Kingdom"
   }
dataset['country'] = dataset['country'].replace(country_map)
dataset['country'].unique()

array(['Norway', 'Kenya', 'Uganda', 'South Africa', 'Denmark',
       'Netherlands', 'Italy', 'Spain', 'United Kingdom', 'Somali',
       'Nigeria', 'Germany', 'France'], dtype=object)

***

## 2. Correcting Residence Type Variations
    Residence type variations
    e.g. BI-Residence, BIResidence, BI_Residence → unify to BI Residence

In [26]:
def categorize_residence(residence):
  first_two = str(residence).strip()[:2].upper()
  if first_two == 'PR':
      return 'Private'
  elif first_two == 'BI':
      return 'BI-Residence'
  elif first_two == 'SO':
      return 'Sognsvann'
  else:
      return 'Other'
dataset['residence'] = dataset['residence'].apply(categorize_residence)
dataset.head()

,fNAME,lNAME,Age,gender,country,residence,entryEXAM,prevEducation,studyHOURS,Python,DB
0,Christina,Binger,44,Female,Norway,Private,72,Masters,158,59.0,55
1,Alex,Walekhwa,60,M,Kenya,Private,79,Diploma,150,60.0,75
2,Philip,Leo,25,Male,Uganda,Sognsvann,55,HighSchool,130,74.0,50
3,Shoni,Hlongwane,22,F,South Africa,Sognsvann,40,High School,120,NaN,44
4,Maria,Kedibone,23,Female,South Africa,Sognsvann,65,High School,122,91.0,80


***

## 3. Correcting Education Level Typos and Casing Issues

    Education level typos and casing issues
    e.g. Barrrchelors → Bachelor, DIPLOMA, Diplomaaa → Diploma

In [29]:
def categorize_education(education):
  first_two = str(education).strip()[:2].upper()
  if first_two == 'HI':
      return 'High School'
  elif first_two == 'DI':
      return 'Diploma'
  elif first_two == 'BA':
      return 'Bachelors'
  elif first_two == 'MA':
      return 'Masters'
  else:
      return 'Doctorate'
dataset['prevEducation'] = dataset['prevEducation'].apply(categorize_education)
dataset['prevEducation'].unique()

array(['Masters', 'Diploma', 'High School', 'Bachelors', 'Doctorate'],
      dtype=object)

***

## 4. Correcting Gender Value Noise

  Gender value noise
    e.g. M, F, female → standardize to Male / Female

In [32]:
def correct_gender(gender):
  gender = str(gender).strip()[:1].upper()
  if gender in ["M" , ' Male']:
    return 'Male'
  else:
    return 'Female'
dataset['gender'] =dataset['gender'].apply(correct_gender)
dataset['gender'].unique()

array(['Female', 'Male'], dtype=object)

***

## 5. Correcting Missing Scores in Python Subject
 Missing scores in Python subject
    Fill NaN values using column mean or suitable imputation strategy

In [35]:
mean_score = dataset['Python'].mean()
dataset['Python'].fillna(mean_score, inplace= True)
dataset['Python'] = dataset['Python'].round(1)

***

## 6. Convert all numerical Values In the Datset to Type Float

In [38]:
dataset = dataset.apply(pd.to_numeric, errors='ignore').astype(float, errors='ignore')
unique_cols = dataset.columns.unique()

***

### Other Preprocessing Tasks

In [41]:
#Make a copy of the data after cleaning
original_dataset = dataset.copy(deep=True)


In [42]:
# Convert Categorical Variables Using One-Hot Encoding
categorical_cols = ['gender','prevEducation']
dataset = pd.get_dummies(dataset, columns = categorical_cols)

In [43]:
# Make a copy of the data after One-Hot Encoding
original_dataset2 = dataset.copy(deep=True)


***

# Key Strategic Areas

You are required to analyze and provide actionable insights for the following three areas:
1. Admissions Optimization

    Should entry exams remain the primary admissions filter?

Your task is to evaluate the predictive power of entry exam scores compared to other features such as prior education, age, gender, and study hours.

✅ Deliverables:

    Feature importance ranking for predicting Python and DB scores
    Admission policy recommendation (e.g., retain exams, add screening tools, adjust thresholds)
    Business rationale and risk analysis


## Train the Model using RandomForest as it gives each features importance

***

### Feature Ranking for DB Scores

***

In [50]:
def grouped_dummmy_importance(feature_importances, dummy_sep="_"):
    grouped = {}
    for col, importance in feature_importances.items():
        if dummy_sep in col:
            base_name = col.split(dummy_sep)[0]
            grouped[base_name] = grouped.get(base_name, 0) + importance
        else:
            grouped[col] = importance
    return pd.Series(grouped).sort_values(ascending=False)


x = dataset[['Age','studyHOURS', 'gender_Male','gender_Female', 'prevEducation_Diploma',
       'prevEducation_Doctorate', 'prevEducation_High School', 'prevEducation_Bachelors',
       'prevEducation_Masters', 'entryEXAM']]
y_DB = dataset['DB']

rf_DB = RandomForestRegressor(n_estimators=300, oob_score=True, random_state=42)
rf_DB.fit(x, y_DB)

DB_importance = pd.Series(rf_DB.feature_importances_, index=x.columns)


grouped_importance_DB = grouped_dummmy_importance(DB_importance)
grouped_importance_DB

entryEXAM        0.634955
Age              0.119891
studyHOURS       0.107782
gender           0.078695
prevEducation    0.058678
dtype: float64

***

### Feature Ranking for Python Scores

In [53]:
x = dataset[['Age','studyHOURS', 'gender_Male','gender_Female', 'prevEducation_Diploma',
       'prevEducation_Doctorate', 'prevEducation_High School', 'prevEducation_Bachelors',
       'prevEducation_Masters', 'entryEXAM']]
y_python = dataset['Python']

rf_py = RandomForestRegressor(n_estimators=300, oob_score=True, random_state=42)
rf_py.fit(x, y_python)

py_importance = pd.Series(rf_py.feature_importances_, index=x.columns)

grouped_importance_py = grouped_dummmy_importance(py_importance)
grouped_importance_py

studyHOURS       0.687634
entryEXAM        0.204046
Age              0.073728
prevEducation    0.018868
gender           0.015723
dtype: float64

***

### Combined Rankings for both Python and DB

In [56]:
combined = pd.DataFrame({
    "Python Importance": grouped_importance_py,
    "DB Importance": grouped_importance_DB
}).round(4)

print(combined)


               Python Importance  DB Importance
Age                       0.0737         0.1199
entryEXAM                 0.2040         0.6350
gender                    0.0157         0.0787
prevEducation             0.0189         0.0587
studyHOURS                0.6876         0.1078


***

### Actionable Insights for Admissions Optimization

Looking at the results, Python peformance is mostly driven by study hours (68.8%), while DB perfomance is mostly driven by entry exams (63.5%). In both cases, age, gender and prior eduction tend to be very weak predictors.


Therefore, it is recommended to use a hybrid admissions process that considers both entry exams for DB success and studyhours for python performance.As an option, it may be prudent to introduce a short pre-course trial that assesses study behaviour among learners. This will ensure the admission process is fairer and more data-driven, while also strengthening admission quality.
Moreover, continued reliance on entry exam alone suggests that thereis a risk of admitting high-exam scorers who wont study due to the weak prediction for python performance observed (20.4%) . Also, although study hours has strong prediction for Python perfomance, it poorly predicts DB (10.8%) therefore posing a risk of admitting motivated but unprepared students. 
As such, the best approach remains considering a combination of the two for an accurate admision process. 


***

# 2. Curriculum Support Strategy

    Are there at-risk student groups who need extra support?

Your task is to uncover whether certain backgrounds (e.g., prior education level, country, residence type) correlate with poor performance and recommend targeted interventions.

✅ Deliverables:

    At-risk segment identification
    Support program design (e.g., prep course, mentoring)
    Expected outcomes, costs, and KPIs


***

### Means for the Various Groups

***

### ANOVA for the Various Groups

In [66]:
def anova_summary(original_dataset, categorical_vars,target_vars):
    results = []   

    for target in target_vars:
        for cat_var in categorical_vars:
            groups = original_dataset[cat_var].dropna().unique()
            scores_by_group = [original_dataset[original_dataset[cat_var] == group][target] for group in groups]

            f_stat, p_val = f_oneway(*scores_by_group)
            results.append({
                "Target": target,
                "Variable": cat_var,
                "F_statistic": f_stat,
                "P_value": p_val})

    summary_df = pd.DataFrame(results).sort_values(['Target', 'P_value']).reset_index(drop=True)

    return summary_df

categorical_vars = ['prevEducation', 'country', 'residence']
target_vars = ['Python', 'DB']

summary = anova_summary(original_dataset, categorical_vars,target_vars)

significant= summary[summary['P_value'] < 0.05]
significant


,Target,Variable,F_statistic,P_value


***

### At-Risk Determination

In [69]:
def at_risk_groups(dataset, categorical_vars, target_vars, threshold=60):
    results = []
    for cat_var in categorical_vars:
        for target in target_vars:
            group_means = dataset.groupby(cat_var)[target].mean().reset_index()
            group_means['At_Risk'] = group_means[target] < threshold
            group_means['Target'] = target
            group_means.rename(columns={cat_var: 'Group', target: 'Average_Score'}, inplace=True)
            group_means = group_means[group_means['At_Risk']]
            results.append(group_means)

    return pd.concat(results, ignore_index=True)

categorical_vars = ['prevEducation', 'country', 'residence']
target_vars = ['Python', 'DB']
at_risk_summary = at_risk_groups(original_dataset, categorical_vars, target_vars)

print(at_risk_summary)


     Group  Average_Score  At_Risk  Target
0  Nigeria           51.0     True  Python
1   France           44.0     True      DB
2    Spain           58.5     True      DB


***

### Actionable Insights for Curriculum Support Strategy

ANOVA and an at-risk metric that identifies student groups that fall belowa certain threshold (60% was considered in this case as a placeholder) was considered.   
For ANOVA, there is no statistically significant differences between the group means although the different group means show some levelof variation.
Considering at-risk segement identification using a threshold of 60%, all groups performed well regardless of their previous education, which is also the case for residence types.

However, a few countries show notably low mean scores like Nigeria (51%) for python, and France (44%) and Spain (58.5%) for DB. Hence, targeted prep courses and resource access should be arranged for Nigerian students taking python and French and Spanish students taking DB.


***

# 3. Resource Allocation & Program ROI

    How can we allocate resources for maximum student success?

Your task is to segment students by success profiles and suggest differentiated teaching/facility strategies.

✅ Deliverables:

    Performance drivers
    Student segmentation
    Resource allocation plan and ROI projection


***

### Creating Age and StudyHours Segments

In [77]:
bins = [20,30, 40, 50, 60, 70, 80]
labels = ['21-29', '30-39', '40-49', '50-59', '60-69', '70-79']
original_dataset2['Age_Group'] = pd.cut(original_dataset2['Age'], bins=bins, labels=labels, right=False)

In [78]:
bins = [110, 125, 140, 155, 170]
labels = ['110-124', '125-139', '140-154', '155-170' ]
original_dataset2['StudyHours_Group'] = pd.cut(original_dataset2['studyHOURS'], bins=bins, labels=labels,right=False)

In [79]:
original_dataset2[['StudyHours_Group','Age_Group' ]].head()

,StudyHours_Group,Age_Group
0,155-170,40-49
1,140-154,60-69
2,125-139,21-29
3,110-124,21-29
4,110-124,21-29


***

### ANOVA for the Various Segments

In [82]:

def run_anova (df, category, target):
    groups = [group[target].dropna() for _, group in original_dataset2.groupby(category)]
    if len(groups) <= 1:
        return None, None

    f_stat, p_value = stats.f_oneway(*groups)
    return f_stat,p_value

categorical_vars = ['Age_Group', 'StudyHours_Group', 'gender_Female', 'gender_Male', 'prevEducation_Bachelors', 'prevEducation_Diploma',
       'prevEducation_Doctorate', 'prevEducation_High School',
       'prevEducation_Masters']
target_vars = ['Python', 'DB']

anova_results = []

for cat in categorical_vars:
    for tgt in target_vars:
        f,p = run_anova(dataset, cat, tgt)
        anova_results.append([cat, tgt,f, p])

anova_df = pd.DataFrame(anova_results, columns =['Variable', 'Target', 'F_Statistic', 'P_value'])

significant= anova_df[anova_df['P_value'] < 0.05]
significant


,Variable,Target,F_Statistic,P_value
0,Age_Group,Python,2.506706,3.791601e-02
2,StudyHours_Group,Python,36.241888,1.874604e-14
3,StudyHours_Group,DB,5.663260,1.521614e-03
5,gender_Female,DB,6.298553,1.424040e-02
7,gender_Male,DB,6.298553,1.424040e-02
14,prevEducation_High School,Python,4.430049,3.866134e-02
15,prevEducation_High School,DB,5.998961,1.664885e-02
17,prevEducation_Masters,DB,4.183460,4.432803e-02


***

### ANOVA for Age Groups

In [85]:
def anova_age_group(df, target):
    means = df.groupby('Age_Group')[target].mean().reset_index()
    means.columns = ['Age_Group', 'Mean_Score']

    groups = [group[target].dropna() for _, group in df.groupby('Age_Group')]
    
    if len(groups) <= 1:
        means["F_Statistic"] = None
        means["P_value"] = None
        means["Target"] = target
        
        return means
        
    f_stat, p_value = stats.f_oneway(*groups)
    means["F_Statistic"] = f_stat
    means["P_value"] = p_value
    means["Target"] = target

    return means


age_python = anova_age_group(original_dataset2, 'Python')
age_DB = anova_age_group(original_dataset2, 'DB')

anova_age_table = pd.concat([age_python, age_DB], ignore_index=True)
anova_age_table
significant= anova_age_table[anova_age_table['P_value'] < 0.05]
significant

,Age_Group,Mean_Score,F_Statistic,P_value,Target
0,21-29,72.992308,2.506706,0.037916,Python
1,30-39,77.428571,2.506706,0.037916,Python
2,40-49,80.176471,2.506706,0.037916,Python
3,50-59,78.000000,2.506706,0.037916,Python
4,60-69,73.500000,2.506706,0.037916,Python
5,70-79,31.000000,2.506706,0.037916,Python


***

### ANOVA for StudyHour Groups

In [88]:
def anova_studyhours_group(df, target):
    means = df.groupby('StudyHours_Group')[target].mean().reset_index()
    means.columns = ['StudyHours_Group', 'Mean_Score']

    groups = [group[target].dropna() for _, group in df.groupby('StudyHours_Group')]
   
    if len(groups) <= 1:
        means["F_Statistic"] = None
        means["P_value"] = None
        means["Target"] = target
        
        return means
        
    f_stat, p_value = stats.f_oneway(*groups)
    means["F_Statistic"] = f_stat
    means["P_value"] = p_value
    means["Target"] = target

    return means


studyhours_python = anova_studyhours_group(original_dataset2, 'Python')
studyhours_DB = anova_studyhours_group(original_dataset2, 'DB')

anova_age_table = pd.concat([studyhours_python, studyhours_DB], ignore_index=True)
anova_age_table
significant= anova_age_table[anova_age_table['P_value'] < 0.05]
significant

,StudyHours_Group,Mean_Score,F_Statistic,P_value,Target
0,110-124,46.112500,36.241888,1.874604e-14,Python
1,125-139,69.600000,36.241888,1.874604e-14,Python
2,140-154,70.883333,36.241888,1.874604e-14,Python
3,155-170,83.652174,36.241888,1.874604e-14,Python
4,110-124,55.625000,5.663260,1.521614e-03,DB
5,125-139,56.800000,5.663260,1.521614e-03,DB
6,140-154,64.833333,5.663260,1.521614e-03,DB
7,155-170,75.065217,5.663260,1.521614e-03,DB


***

### At-Risk Determination

In [91]:
def at_risk_groups(dataset, categorical_vars, target_vars, threshold= 60):
    results = []

    # Mapping dummy columns to readable names
    dummy_label_map = {
        'gender_Female': 'Female',
        'gender_Male': 'Male',
        'prevEducation_Bachelors': 'Bachelors',
        'prevEducation_Diploma': 'Diploma',
        'prevEducation_Doctorate': 'Doctorate',
        'prevEducation_High School': 'High School',
        'prevEducation_Masters': 'Masters'
    }

    for cat_var in categorical_vars:
        for target in target_vars:
            df = dataset.copy()

            # For dummy variables, create a column with the label only if True
            if cat_var in dummy_label_map:
                label = dummy_label_map[cat_var]
                df['Group'] = df[cat_var].apply(lambda x: label if x else pd.NA)
            else:
                df['Group'] = df[cat_var]

            # Remove rows where Group is missing
            df = df.dropna(subset=['Group'])

            # Compute group means
            group_means = df.groupby('Group')[target].mean().reset_index()
            group_means['At_Risk'] = group_means[target] < threshold
            group_means['Target'] = target
            group_means.rename(columns={target: 'Average_Score'}, inplace=True)

            # Keep only at-risk groups
            group_means = group_means[group_means['At_Risk']]

            results.append(group_means)

    return pd.concat(results, ignore_index=True)

categorical_vars = [
    'Age_Group', 'StudyHours_Group',
    'gender_Female', 'gender_Male',
    'prevEducation_Bachelors', 'prevEducation_Diploma',
    'prevEducation_Doctorate', 'prevEducation_High School',
    'prevEducation_Masters'
]
target_vars = ['Python', 'DB']

at_risk_summary = at_risk_groups(original_dataset2, categorical_vars, target_vars)
print(at_risk_summary)



     Group  Average_Score  At_Risk  Target
0    70-79        31.0000     True  Python
1    70-79        42.0000     True      DB
2  110-124        46.1125     True  Python
3  110-124        55.6250     True      DB
4  125-139        56.8000     True      DB


***

### Actionable Insights for Curriculum Support Strategy

Among the performance drivers that were considered include study hours, prior education, gender and age.
For age and study hours, the learners were categorised in bins of various age groups as per their age (e.g 20-29) and studyhour groups accoriding to the number of hours in which they studied (e.g 155-177). 

Therefore, the leraners can be segmented into various categories based on study hours, prior education, gender and age.

Both ANOVA and means were calculated and only groups that showed statistically significant differences in their means and had means below the threshold of 60% and considered at_risk were proposed for resource allocation plan. 

These segments included learners aged 70-79 who had low scores on both categories (Python-31.0%, DB-42.0%). Those who studied for 110-124 hours also scored lower marks (Python-46.1%, DB-55.6%) while those who studied for 125-139 hours only had a low score for DB - 56.8%.

Age 70–79 & Low Study Hours (110–124) will require intensive support like small-group instruction, 1:1 mentoring, lab assistance, and extra tutoring sessions.

Similarly, low study hours (125–139) may also need targeted skill-building sessions and additional practice resources.

For the other groups, there should be minimal intervention with standard lectures, optional advanced modules, peer study groups while also encouraging self-directed learning and enrichment activities, without allocating intensive resources.
